# Two-Phase ArcFace Training

Phase 1 trains Sub-center ArcFace (K=3 sub-centers per identity), then automatically flags noisy samples based on sub-center assignment consistency. Phase 2 retrains standard ArcFace on the cleaned dataset.

Set `PHASE = "1"`, `"2"`, or `"both"`.

## Configuration

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA_ROOT     = "/content/data/dataset_a" if IN_COLAB else "./datasets/dataset_a"
SAVE_DIR      = "/content/checkpoints"    if IN_COLAB else "./checkpoints"
PHASE         = "both"    # "1", "2", or "both"
SCHEDULE      = "step"    # "step" or "cosine"
EMBEDDING_DIM = 512       # 128, 256, or 512
EPOCHS        = 35
BATCH_SIZE    = 128
NUM_WORKERS  = 0  # >0 causes multiprocessing errors in Jupyter on macOS

DEBUG         = False     # True: 500-image subset, 2 epochs

import torch
# Auto-detects: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

In [ ]:
import math
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from tqdm import tqdm

## Device Setup

In [ ]:
device = torch.device(DEVICE)
print(f"Device: {device}")

## Model Definition (FaceEncoder)

In [ ]:
class FaceEncoder(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.fc = nn.Linear(2048, embedding_dim)
        self._embedding_dim = embedding_dim

    @property
    def embedding_dim(self):
        return self._embedding_dim

    def forward(self, x):
        return self.backbone(x)

    def encode(self, x):
        return F.normalize(self.forward(x), p=2, dim=1)

## Custom Transforms

In [ ]:
class GridMask:
    def __init__(self, grid=4, drop_ratio=0.25, p=0.15):
        self.grid = grid
        self.n_drop = math.ceil(drop_ratio * grid * grid)
        self.p = p

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor
        _, H, W = tensor.shape
        cell_h = H // self.grid
        cell_w = W // self.grid
        cells = [(r, c) for r in range(self.grid) for c in range(self.grid)]
        drop = random.sample(cells, self.n_drop)
        tensor = tensor.clone()
        for r, c in drop:
            tensor[:, r * cell_h:(r + 1) * cell_h, c * cell_w:(c + 1) * cell_w] = 0.0
        return tensor


TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomResizedCrop(112, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
    GridMask(grid=4, drop_ratio=0.25, p=0.15),
])

## Dataset

In [ ]:
class FaceTrainDataset(Dataset):
    def __init__(self, root, parquet_file="test.parquet"):
        self.root = root
        df = pd.read_parquet(os.path.join(root, parquet_file))
        unique_templates = sorted(df["template_id"].unique())
        self.tid_to_label = {tid: i for i, tid in enumerate(unique_templates)}
        self.num_classes = len(unique_templates)
        self.image_paths = df["image_path"].tolist()
        self.labels = [self.tid_to_label[tid] for tid in df["template_id"].values]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = os.path.join(self.root, self.image_paths[idx])
        img = Image.open(path).convert("RGB")
        return TRAIN_TRANSFORM(img), self.labels[idx]

## Loss Functions

In [ ]:
class SubCenterArcFace(nn.Module):
    def __init__(self, embedding_dim, num_classes, K=3, s=64.0, m=0.5):
        super().__init__()
        self.K = K
        self.s = s
        self.m = m
        self.num_classes = num_classes
        self.weight = nn.Parameter(torch.FloatTensor(num_classes * K, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, emb, labels):
        B = emb.size(0)
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))  # (B, C*K)
        cosine = cosine.view(B, self.num_classes, self.K).max(dim=2).values  # (B, C)
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.unsqueeze(1), 1.0)
        logits = torch.cos(theta + self.m * one_hot) * self.s
        return F.cross_entropy(logits, labels)

    def sub_center_cosines(self, emb):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))
        return cosine.view(emb.size(0), self.num_classes, self.K)


class ArcFaceLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, s=64.0, m=0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, emb, labels):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.unsqueeze(1), 1.0)
        logits = torch.cos(theta + self.m * one_hot) * self.s
        return F.cross_entropy(logits, labels)

## Training Helpers

In [ ]:
def make_optimizer(model, criterion, lr=0.1, wd=5e-4):
    params = list(model.parameters()) + list(criterion.parameters())
    return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd)


def make_scheduler(optimizer, schedule, epochs):
    if schedule == "step":
        return torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[20, 28, 32], gamma=0.1)
    return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )


def make_phase2_scheduler(optimizer, schedule):
    if schedule == "step":
        return torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[8, 12, 14], gamma=0.1)
    return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2, eta_min=1e-6
    )


def run_epoch(model, criterion, loader, optimizer, scheduler, device, desc):
    model.train()
    total = 0.0
    pbar = tqdm(loader, desc=desc)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(model(imgs), labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total / len(loader)


def save_ckpt(path, model, epoch, loss, embedding_dim):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "loss": loss,
        "embedding_dim": embedding_dim,
    }, path)

## Phase 1 — Sub-center ArcFace

In [ ]:
def phase1(data_root, save_dir, embedding_dim, batch_size, num_workers, schedule, epochs, device, dataset):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    model = FaceEncoder(embedding_dim).to(device)
    criterion = SubCenterArcFace(embedding_dim, dataset.num_classes).to(device)
    optimizer = make_optimizer(model, criterion)
    scheduler = make_scheduler(optimizer, schedule, epochs)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        num_workers=num_workers, pin_memory=(device.type == 'cuda'), drop_last=True)

    for ep in range(epochs):
        avg = run_epoch(model, criterion, loader, optimizer, scheduler, device, f"Ph1 {ep+1}/{epochs}")
        scheduler.step()
        print(f"  epoch {ep+1} loss={avg:.4f}")

    ckpt_path = save_dir / "phase1.pth"
    save_ckpt(ckpt_path, model, epochs - 1, avg, embedding_dim)
    print(f"Phase 1 checkpoint: {ckpt_path}")

    # Noise isolation
    print("Running noise isolation...")
    model.eval()
    sub_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=(device.type == 'cuda'))
    assignments = []
    with torch.inference_mode():
        for imgs, labels in tqdm(sub_loader, desc="Sub-center assignment"):
            cos = criterion.sub_center_cosines(model(imgs.to(device)))
            for i, lab in enumerate(labels.tolist()):
                best_k = cos[i, lab].argmax().item()
                assignments.append((lab, best_k))

    identity_sub = {}
    for lab, k in assignments:
        identity_sub.setdefault(lab, []).append(k)
    dominant = {lab: Counter(ks).most_common(1)[0][0] for lab, ks in identity_sub.items()}

    noise_flags = np.array([
        assignments[i][1] != dominant[assignments[i][0]]
        for i in range(len(assignments))
    ])
    flags_path = save_dir / "noise_flags.npy"
    np.save(flags_path, noise_flags)
    n_noise = noise_flags.sum()
    print(f"Flagged {n_noise}/{len(noise_flags)} samples as noise ({100*n_noise/len(noise_flags):.1f}%)")

    return str(ckpt_path), str(flags_path)

## Phase 2 — Standard ArcFace

In [ ]:
def phase2(data_root, save_dir, embedding_dim, batch_size, num_workers, schedule, device, dataset, flags_path=None, init_ckpt=None):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    if flags_path and os.path.exists(flags_path):
        noise_flags = np.load(flags_path)
        clean_indices = np.where(~noise_flags)[0].tolist()
        clean_dataset = Subset(dataset, clean_indices)
        print(f"Phase 2: {len(clean_dataset)} clean samples (dropped {noise_flags.sum()} noisy)")
    else:
        clean_dataset = dataset
        print("Phase 2: no noise_flags.npy found, using full dataset")

    model = FaceEncoder(embedding_dim).to(device)
    if init_ckpt and os.path.exists(init_ckpt):
        ckpt = torch.load(init_ckpt, map_location=device, weights_only=True)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Initialized from {init_ckpt}")

    criterion = ArcFaceLoss(embedding_dim, dataset.num_classes).to(device)
    optimizer = make_optimizer(model, criterion)
    scheduler = make_phase2_scheduler(optimizer, schedule)

    loader = DataLoader(clean_dataset, batch_size=batch_size, shuffle=True,
                        num_workers=num_workers, pin_memory=(device.type == 'cuda'), drop_last=True)

    epochs = 15
    for ep in range(epochs):
        avg = run_epoch(model, criterion, loader, optimizer, scheduler, device, f"Ph2 {ep+1}/{epochs}")
        scheduler.step()
        print(f"  epoch {ep+1} loss={avg:.4f}")

    ckpt_path = save_dir / "phase2.pth"
    save_ckpt(ckpt_path, model, epochs - 1, avg, embedding_dim)
    print(f"Phase 2 checkpoint: {ckpt_path}")
    return str(ckpt_path)

## Dataset Loading

In [ ]:
dataset = FaceTrainDataset(DATA_ROOT)
if DEBUG:
    full_num_classes = dataset.num_classes
    dataset = Subset(dataset, list(range(min(500, len(dataset)))))
    dataset.num_classes = full_num_classes
    EPOCHS = 2
    print("Debug mode: 500 samples, 2 epochs")

print(f"Dataset: {len(dataset)} images, {dataset.num_classes} identities")

## Run Training

In [ ]:
flags_path = None
init_ckpt = None

if PHASE in ("1", "both"):
    init_ckpt, flags_path = phase1(
        DATA_ROOT, SAVE_DIR, EMBEDDING_DIM, BATCH_SIZE, NUM_WORKERS,
        SCHEDULE, EPOCHS, device, dataset
    )

if PHASE in ("2", "both"):
    phase2(
        DATA_ROOT, SAVE_DIR, EMBEDDING_DIM, BATCH_SIZE, NUM_WORKERS,
        SCHEDULE, device, dataset, flags_path=flags_path, init_ckpt=init_ckpt
    )